# Variant 0–5 evaluation metrics EDA

This notebook reads only the standardized `evaluation_metrics` output. Legacy hallucination metrics are deliberately ignored. Missing or architecturally inapplicable values remain `NaN`; they are never converted to zero.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
OUTPUT_DIR = Path('results/eda_exports')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

The standard full-run filenames are listed below. Missing files are skipped. Until the full runs exist, the clean Variant 3 smoke result is used as a fallback.

In [ ]:
RESULT_FILES = {
    variant: Path(f'results/eval_variant_{variant}_test_2_metrics.json')
    for variant in range(6)
}
if not any(path.exists() for path in RESULT_FILES.values()):
    RESULT_FILES = {3: Path('results/eval_variant_3_test_2_metrics_limit1_clean.json')}

DATA_FILE = Path('splits/lite_7db_test_2.jsonl')
existing = {v: p for v, p in RESULT_FILES.items() if p.exists()}
assert existing, 'No configured result files were found.'
display(pd.DataFrame([{'variant': f'Variant {v}', 'file': str(p)} for v, p in existing.items()]))

## Load and validate

This cell rejects files without the new evaluation block and reports any legacy keys that remain in an input file.

In [ ]:
def load_json(path):
    with path.open() as handle:
        return json.load(handle)

runs = {variant: load_json(path) for variant, path in existing.items()}
audit = []
for variant, run in runs.items():
    results = run.get('results', [])
    legacy = ('hallucination' in run.get('metrics', {}) or
              any('hallucination_metrics' in result for result in results))
    missing = sum('evaluation_metrics' not in result for result in results)
    audit.append({
        'variant': f'Variant {variant}', 'tasks': len(results),
        'legacy_metrics_present': legacy,
        'tasks_missing_evaluation_metrics': missing,
    })
audit_df = pd.DataFrame(audit)
display(audit_df)
assert not audit_df['legacy_metrics_present'].any(), 'A configured file still contains legacy metrics.'
assert not audit_df['tasks_missing_evaluation_metrics'].any(), 'Some tasks lack evaluation_metrics.'

## Build task-level analysis tables

In [ ]:
task_lookup = {}
if DATA_FILE.exists():
    for line in DATA_FILE.read_text().splitlines():
        if line.strip():
            task = json.loads(line)
            task_lookup[str(task.get('instance_id', task.get('id')))] = task

def structural_class(sql_value, category):
    text = ' '.join(sql_value) if isinstance(sql_value, list) else str(sql_value or '')
    clean = re.sub(r'/\*.*?\*/|--[^\n]*', ' ', text, flags=re.S).upper()
    nested = bool(re.search(r'\bWITH\b|\(\s*SELECT\b|\b(?:UNION|INTERSECT|EXCEPT)\b', clean))
    joined = bool(re.search(r'\bJOIN\b', clean))
    if category == 'Management':
        procedural = bool(re.search(r'\b(?:FUNCTION|PROCEDURE|TRIGGER)\b|\bDO\s+\$\$', clean))
        if procedural or nested: return 'nested_complex'
        if joined: return 'non_nested_complex'
        return 'easy'
    if nested: return 'nested_complex'
    if joined: return 'non_nested_complex'
    return 'easy'

def nested_get(mapping, *keys):
    value = mapping
    for key in keys:
        if not isinstance(value, dict): return np.nan
        value = value.get(key)
    return np.nan if value is None else value

common_rows, phase_rows, harness_rows, multi_rows = [], [], [], []
for variant, run in runs.items():
    for result in run.get('results', []):
        task_id = str(result.get('instance_id', result.get('task_id', 'unknown')))
        task = task_lookup.get(task_id, {})
        metrics = result['evaluation_metrics']
        common = metrics['common']
        base = {
            'variant_number': variant, 'variant': f'Variant {variant}',
            'task_id': task_id,
            'database': task.get('selected_database', result.get('database')),
            'category': task.get('category', 'Unknown'),
        }
        base['structure'] = structural_class(task.get('sol_sql'), base['category'])
        invalid = common['invalid_references']
        common_rows.append({**base,
            'task_success': common['task_success'],
            'submission_present': common['submission_present'],
            'table_f1': common['table']['f1'],
            'column_f1': common['column']['f1'],
            'join_path_f1': common['join_path']['f1'],
            'kb_f1': common['kb']['f1'],
            'overall_invalid_reference_rate': invalid['overall_invalid_reference_rate'],
            'invalid_table_rate': invalid['invalid_table_rate'],
            'invalid_column_rate': invalid['invalid_column_rate'],
            'invalid_kb_rate': invalid['invalid_kb_rate'],
            'task_has_invalid_reference': invalid['task_has_invalid_reference'],
            'structural_f1': common['final_sql_structural_agreement']['f1'],
            'steps': common['steps_used'], 'tokens': common['total_tokens'],
            'latency_seconds': common['elapsed_seconds'],
        })
        phase = metrics['phase_specific']
        if phase.get('applicable'):
            pre, plan, generation, post = (phase[k] for k in ('preprocessing','query_planning','sql_generation','postprocessing'))
            phase_rows.append({**base,
                'pre_completion': pre['completion'], 'pre_table_f1': pre['table']['f1'],
                'pre_column_f1': pre['column']['f1'], 'pre_join_f1': pre['join_path']['f1'],
                'pre_kb_f1': pre['kb']['f1'], 'complete_grounding': pre['complete_grounding'],
                'plan_completion': plan['completion'], 'first_plan_acceptance': plan['first_attempt_accepted'],
                'plan_attempts': plan['attempts'],
                'query_structural_f1': nested_get(plan, 'query_structural_agreement', 'f1'),
                'management_contract_complete': plan['management_contract_complete'],
                'kb_formula_preservation': nested_get(plan, 'kb_formula_preservation', 'preservation_rate'),
                'first_validation_pass': generation['first_validation_passed'],
                'validation_attempts': generation['validation_attempts'],
                'rejection_reasons': '|'.join(generation.get('rejection_reasons', [])),
                'post_eligible': post['eligible'], 'correction_attempted': post['correction_attempted'],
                'correction_attempts': post['correction_attempts'], 'resolved': post['resolved'],
                'recovered_task': post['recovered_task'],
            })
        harness = metrics['harness_specific']
        if harness.get('applicable'):
            harness_rows.append({**base, **{k: v for k, v in harness.items() if k != 'applicable'}})
        multi = metrics['multi_agent_specific']
        if multi.get('applicable'):
            multi_rows.append({**base, **{k: v for k, v in multi.items() if k != 'applicable'}})

common_df = pd.DataFrame(common_rows)
phase_df = pd.DataFrame(phase_rows)
harness_df = pd.DataFrame(harness_rows)
multi_df = pd.DataFrame(multi_rows)
display(common_df.head())

## Headline evaluation table and plots

In [ ]:
headline = common_df.groupby('variant', sort=False).agg(
    tasks=('task_id','size'), task_success_rate=('task_success','mean'),
    submission_rate=('submission_present','mean'), table_f1=('table_f1','mean'),
    column_f1=('column_f1','mean'), join_path_f1=('join_path_f1','mean'),
    kb_f1=('kb_f1','mean'), invalid_reference_rate=('overall_invalid_reference_rate','mean'),
    structural_f1=('structural_f1','mean'), average_steps=('steps','mean'),
    average_tokens=('tokens','mean'), average_latency_seconds=('latency_seconds','mean'),
).reset_index()
display(headline.style.format({c:'{:.1%}' for c in ['task_success_rate','submission_rate','table_f1','column_f1','join_path_f1','kb_f1','invalid_reference_rate','structural_f1']}))

rate_metrics = ['task_success_rate','submission_rate','table_f1','column_f1','join_path_f1','kb_f1','structural_f1','invalid_reference_rate']
ax = headline.set_index('variant')[rate_metrics].T.plot.bar(figsize=(14,6))
ax.set_ylim(0,1.05); ax.tick_params(axis='x', rotation=35); ax.set_ylabel('Rate'); ax.set_title('Headline metrics by variant')
plt.tight_layout(); plt.show()

## Success, invalid references, cost and latency

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15,10))
common_df.pivot_table(index='category',columns='variant',values='task_success',aggfunc='mean').plot.bar(ax=axes[0,0])
axes[0,0].set_title('Success rate by task type'); axes[0,0].set_ylim(0,1.05)
common_df.pivot_table(index='structure',columns='variant',values='task_success',aggfunc='mean').plot.bar(ax=axes[0,1])
axes[0,1].set_title('Success rate by SQL structure'); axes[0,1].set_ylim(0,1.05)
common_df.pivot_table(index='database',columns='variant',values='task_success',aggfunc='mean').plot.bar(ax=axes[1,0])
axes[1,0].set_title('Success rate by database'); axes[1,0].tick_params(axis='x', rotation=30)
cost = common_df.groupby('variant', sort=False)[['steps','tokens','latency_seconds']].mean().reset_index()
cost.set_index('variant')['tokens'].plot.bar(ax=axes[1,1], color='#4C78A8')
axes[1,1].set_title('Average total tokens per task')
plt.tight_layout(); plt.show()

invalid_cols = ['invalid_table_rate','invalid_column_rate','invalid_kb_rate','overall_invalid_reference_rate']
ax = common_df.groupby('variant', sort=False)[invalid_cols].mean().T.plot.bar(figsize=(12,5))
ax.set_ylim(0,1.05); ax.tick_params(axis='x',rotation=25); ax.set_title('Actual invalid-reference rates')
plt.tight_layout(); plt.show()

## Individual variant views

In [ ]:
for variant, subset in common_df.groupby('variant', sort=False):
    fig, axes = plt.subplots(1,3,figsize=(16,4))
    subset.groupby('category')['task_success'].mean().plot.bar(ax=axes[0], color='#4C78A8')
    axes[0].set_title('Success by task type'); axes[0].set_ylim(0,1.05)
    subset.groupby('structure')['task_success'].mean().plot.bar(ax=axes[1], color='#59A14F')
    axes[1].set_title('Success by structure'); axes[1].set_ylim(0,1.05); axes[1].tick_params(axis='x', rotation=25)
    subset.groupby('database')['task_success'].mean().plot.bar(ax=axes[2], color='#F28E2B')
    axes[2].set_title('Success by database'); axes[2].set_ylim(0,1.05); axes[2].tick_params(axis='x', rotation=25)
    fig.suptitle(variant); plt.tight_layout(); plt.show()

## Phase-specific metrics — Variants 1, 3, 4 and 5

In [ ]:
if phase_df.empty:
    print('No phase-specific artifacts in the configured runs.')
else:
    phase_rate_cols = ['pre_completion','pre_table_f1','pre_column_f1','pre_join_f1','pre_kb_f1','complete_grounding','plan_completion','first_plan_acceptance','query_structural_f1','management_contract_complete','kb_formula_preservation','first_validation_pass','correction_attempted','resolved','recovered_task']
    phase_summary = phase_df.groupby('variant', sort=False)[phase_rate_cols + ['plan_attempts','validation_attempts','correction_attempts']].mean().reset_index()
    display(phase_summary)
    groups = {
      'Preprocessing':['pre_completion','pre_table_f1','pre_column_f1','pre_join_f1','pre_kb_f1','complete_grounding'],
      'Planning':['plan_completion','first_plan_acceptance','query_structural_f1','management_contract_complete','kb_formula_preservation'],
      'SQL generation':['first_validation_pass'],
      'Post-processing':['correction_attempted','resolved','recovered_task'],
    }
    fig, axes = plt.subplots(2,2,figsize=(18,11))
    for ax, (title, cols) in zip(axes.flat, groups.items()):
        phase_summary.set_index('variant')[cols].T.dropna(how='all').plot.bar(ax=ax)
        ax.set_title(title); ax.set_ylim(0,1.05); ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); plt.show()
    reasons = phase_df.assign(rejection_reason=phase_df['rejection_reasons'].replace('',np.nan)).dropna(subset=['rejection_reason'])
    if not reasons.empty:
        display(reasons.groupby(['variant','rejection_reason']).size().rename('count').reset_index())

## Harness metrics — Variants 2, 3 and 5

In [ ]:
if harness_df.empty:
    print('No harness-specific artifacts in the configured runs.')
else:
    harness_rates = ['hard_constraint_violation_attempt_rate','post_block_recovery_rate','submission_gate_compliance_rate']
    harness_summary = harness_df.groupby('variant', sort=False)[harness_rates + ['raw_blocked_call_count']].mean().reset_index()
    display(harness_summary)
    ax = harness_summary.set_index('variant')[harness_rates].T.plot.bar(figsize=(11,5))
    ax.set_ylim(0,1.05); ax.tick_params(axis='x',rotation=25); ax.set_title('Harness behaviour')
    plt.tight_layout(); plt.show()
    reason_counts = {}
    for _, row in harness_df.iterrows():
        for reason, count in (row.get('violation_reasons') or {}).items():
            reason_counts[(row['variant'], reason)] = reason_counts.get((row['variant'], reason), 0) + count
    if reason_counts:
        display(pd.DataFrame([{'variant':v,'reason':r,'count':c} for (v,r),c in reason_counts.items()]))

## Multi-agent metrics — Variants 4 and 5

In [ ]:
if multi_df.empty:
    print('No multi-agent artifacts in the configured runs.')
else:
    multi_summary = multi_df.groupby('variant', sort=False).agg(
        end_to_end_completion=('end_to_end_phase_completion','mean'),
        average_backward_transitions=('backward_transitions','mean'),
        handoff_recovery_rate=('handoff_recovery_rate','mean'),
        new_evidence_recovery_rate=('new_evidence_recovery_rate','mean'),
        average_phase_retries=('phase_retries','mean'),
    ).reset_index()
    display(multi_summary)
    shares = []
    for _, row in multi_df.iterrows():
        for agent, share in (row.get('per_agent_token_share') or {}).items():
            shares.append({'variant':row['variant'],'agent':agent,'token_share':share})
    if shares:
        shares_df = pd.DataFrame(shares)
        ax = shares_df.pivot_table(index='agent',columns='variant',values='token_share',aggfunc='mean').plot.bar(figsize=(10,5))
        ax.set_ylim(0,1.05); ax.tick_params(axis='x',rotation=25); ax.set_title('Per-agent token distribution')
        plt.tight_layout(); plt.show()

## Planned pairwise comparisons

The plots are generated only when both variants are loaded.

In [ ]:
for left, right in [(0,1),(0,2),(1,3),(3,4),(4,5)]:
    names = {f'Variant {left}', f'Variant {right}'}
    pair = headline[headline['variant'].isin(names)]
    if len(pair) != 2: continue
    metrics = ['task_success_rate','submission_rate','table_f1','column_f1','join_path_f1','kb_f1','structural_f1','invalid_reference_rate']
    ax = pair.set_index('variant')[metrics].T.plot.bar(figsize=(13,5))
    ax.set_ylim(0,1.05); ax.tick_params(axis='x',rotation=30); ax.set_title(f'Variant {left} vs Variant {right}')
    plt.tight_layout(); plt.show()

## Export analysis tables

In [ ]:
headline.to_csv(OUTPUT_DIR/'headline_metrics.csv',index=False)
common_df.to_csv(OUTPUT_DIR/'task_common_metrics.csv',index=False)
if not phase_df.empty: phase_df.to_csv(OUTPUT_DIR/'task_phase_metrics.csv',index=False)
if not harness_df.empty: harness_df.to_csv(OUTPUT_DIR/'task_harness_metrics.csv',index=False)
if not multi_df.empty: multi_df.to_csv(OUTPUT_DIR/'task_multi_agent_metrics.csv',index=False)
print(f'Exports written to {OUTPUT_DIR.resolve()}')